# ebook2audiobook - Cloud (Google Colab)

Run the whole conversion (and optional translation) on a free cloud GPU.
This notebook receives the settings chosen in the desktop app automatically
(through the Colab URL). You only upload the ebook file here.

## How to use

1. In the desktop app open the **Cloud** tab and click **"Open Google Colab with current settings"**.
2. Runtime -> Change runtime type -> **T4 GPU** (recommended).
3. Run the three cells below top to bottom.
4. Upload your ebook when prompted; the finished audiobook downloads automatically.

If the settings were not picked up automatically, paste the config string
(copied to your clipboard by the app) into the **config_b64** field in the first cell.

In [ ]:
#@title (1) Settings & upload the ebook { display-mode: "form" }
# Reads the settings sent from the desktop app (via the Colab URL) or from the
# manual field below, then picks up the ebook file.
config_b64 = ""  #@param {type:"string"}

import base64, glob, json, os
try:
    from google.colab import output as _cin_output, files as _cin_files
    _IN_COLAB = True
except Exception:
    _IN_COLAB = False

if not config_b64 and _IN_COLAB:
    try:
        import urllib.parse as _up
        _qs = _cin_output.eval_js('window.location.search') or ""
        _params = _up.parse_qs(_qs.lstrip('?'))
        config_b64 = (_params.get('c') or [""])[0]
    except Exception as _e:
        print("Could not read settings from the URL:", _e)

def _decode_cfg(b64):
    if not b64:
        return {}
    b64 = b64.strip()
    b64 += "=" * (-len(b64) % 4)
    return json.loads(base64.urlsafe_b64decode(b64.encode('ascii')).decode('utf-8'))

CFG = _decode_cfg(config_b64)
globals()['CFG'] = CFG
if CFG:
    print("Settings received from the app:")
    print(json.dumps(CFG, ensure_ascii=False, indent=2))
else:
    print("No settings found - defaults will be used. You can paste the config into the field above.")

# The ebook can arrive two ways:
#  a) already uploaded to /content via the Files panel (folder icon on the
#     left -> upload icon) - the reliable way on phones;
#  b) through the upload widget below - fine on desktop, flaky on phones.
EBOOK_EXTS = ('epub', 'mobi', 'azw3', 'azw', 'pdf', 'txt', 'rtf', 'docx',
              'doc', 'html', 'htm', 'fb2', 'md', 'odt')

def _find_ebook():
    found = []
    for _e in EBOOK_EXTS:
        found += glob.glob('/content/*.' + _e)
    return max(found, key=os.path.getmtime) if found else ''

EBOOK = _find_ebook()
if not EBOOK and _IN_COLAB:
    print("\nUpload your ebook file (epub, pdf, mobi, txt, ...).")
    print("PHONE TIP: if the button below is greyed out or says no file was")
    print("chosen, upload the book via the Files panel instead (folder icon")
    print("on the left -> upload icon), then re-run this cell.")
    try:
        _uploaded = _cin_files.upload()
        if _uploaded:
            EBOOK = list(_uploaded.keys())[0]
    except Exception as _e:
        print("Upload widget failed:", _e)
    EBOOK = EBOOK or _find_ebook()

globals()['EBOOK'] = EBOOK
if EBOOK:
    print("\nSelected ebook:", EBOOK)
else:
    print("\nNo ebook found yet - upload it via the Files panel and re-run this cell.")

In [ ]:
#@title (2) Install dependencies & clone the repository
import os, subprocess

CFG = globals().get('CFG', {})
REPO = CFG.get('repo', 'Tarkas/Book-to-audiobook')
BRANCH = CFG.get('branch', 'main')
print(f"Repo: {REPO}  Branch: {BRANCH}")

os_cmds = [
    "sudo apt-get update -qq",
    "apt-get install -y libxcb-cursor0 libegl1 libopengl0",
    "sudo -v && wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin",
    "apt-get install -y ffmpeg espeak-ng mecab libmecab-dev mecab-ipadic-utf8 nodejs",
    "pip install -q mecab-python3 unidic-lite unidic",
    "python -m unidic download",
]
for c in os_cmds:
    print("\n$", c)
    subprocess.run(c, shell=True)

if not os.path.isdir('ebook2audiobook'):
    r = subprocess.run(f"git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git ebook2audiobook", shell=True)
    if r.returncode != 0:
        raise SystemExit("git clone failed - check REPO/BRANCH above.")

# Colab has no wheels for the pinned torch==2.1.0 stack (and ships its own
# CUDA torch anyway), so drop those pins - otherwise pip aborts the WHOLE
# install and nothing gets installed (ModuleNotFoundError: ebooklib etc).
_skip = ('torch==', 'torchaudio==', 'torchvision==')
with open('ebook2audiobook/requirements.txt', encoding='utf-8') as _f:
    _reqs = [l for l in _f if not l.strip().startswith(_skip)]
with open('/content/req_colab.txt', 'w', encoding='utf-8') as _f:
    _f.writelines(_reqs)

print("\nInstalling python requirements (a few minutes)...")
r = subprocess.run("pip install -q -r /content/req_colab.txt", shell=True,
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:])
    print(r.stderr[-5000:])
    raise SystemExit("pip install of requirements FAILED - see errors above.")
print("Requirements installed.")
print("\nSetup done.")

In [ ]:
#@title (3) Convert (and translate) in the cloud + download
import os, glob, subprocess

CFG = globals().get('CFG', {})
EBOOK = globals().get('EBOOK', '')
if not EBOOK or not os.path.exists(EBOOK):
    # Fall back to whatever was dropped into /content via the Files panel
    _exts = ('epub', 'mobi', 'azw3', 'azw', 'pdf', 'txt', 'rtf', 'docx',
             'doc', 'html', 'htm', 'fb2', 'md', 'odt')
    _found = []
    for _e in _exts:
        _found += glob.glob('/content/*.' + _e)
    EBOOK = max(_found, key=os.path.getmtime) if _found else ''
if not EBOOK:
    raise SystemExit("No ebook found. Upload it in cell (1) or via the Files panel, then re-run.")

ebook_path = os.path.abspath(EBOOK)
os.makedirs('/content/out', exist_ok=True)

cmd = [
    "python", "app.py", "--headless", "--script_mode", "full_docker",
    "--ebook", ebook_path,
    "--language", str(CFG.get('language', 'eng')),
    "--output_format", str(CFG.get('output_format', 'm4b')),
    "--device", "gpu",
    "--output_dir", "/content/out",
]

engine = CFG.get('tts_engine')
if engine:
    cmd += ["--tts_engine", str(engine)]

# Resolve the selected voice by name inside the cloned repo
voice_name = CFG.get('voice')
if voice_name:
    matches = glob.glob(f"ebook2audiobook/voices/**/{voice_name}.*", recursive=True)
    if matches:
        cmd += ["--voice", os.path.abspath(matches[0])]
        print("Using voice:", matches[0])
    else:
        print(f"Voice '{voice_name}' not found in repo; using the engine default.")

for key, flag in [('speed', '--speed'), ('temperature', '--temperature'), ('repetition_penalty', '--repetition_penalty')]:
    val = CFG.get(key)
    if val is not None:
        cmd += [flag, str(val)]

if CFG.get('translate'):
    cmd += [
        "--translate",
        "--source_lang", str(CFG.get('source_lang', 'eng')),
        "--target_lang", str(CFG.get('target_lang', CFG.get('language', 'eng'))),
        "--translation_method", str(CFG.get('translation_method', 'google')),
    ]

print("\nRunning:", ' '.join(cmd))
# Stream the conversion log right into the cell (plain subprocess.run output
# is often swallowed by Colab) and keep a copy in /content/conversion.log.
_log = open('/content/conversion.log', 'w', encoding='utf-8')
_p = subprocess.Popen(cmd, cwd='ebook2audiobook', stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1,
                      errors='replace')
for _line in _p.stdout:
    print(_line, end='')
    _log.write(_line)
_p.wait()
_log.close()
print("\napp.py exit code:", _p.returncode)

# Collect and offer the produced audio for download
outputs = []
for ext in ('m4b', 'mp3', 'wav', 'm4a', 'flac', 'ogg', 'aac', 'opus'):
    outputs += glob.glob(f"/content/out/**/*.{ext}", recursive=True)
    outputs += glob.glob(f"ebook2audiobook/audiobooks/**/*.{ext}", recursive=True)
outputs = sorted(set(outputs), key=os.path.getmtime)
if outputs:
    result = outputs[-1]
    print("\nAudiobook ready:", result)
    try:
        from google.colab import files as _files
        _files.download(result)
    except Exception as e:
        print("Auto-download failed:", e, "- find your file at", result)
else:
    print("\nNo audio output found. Last lines of /content/conversion.log:")
    with open('/content/conversion.log', encoding='utf-8', errors='replace') as _f:
        for _line in _f.readlines()[-60:]:
            print(_line, end='')